# MOT20 Tracking Pipeline

End-to-end pipeline: data preparation → YOLO training → evaluation.

**Data split:**
| Split | Sequences | Purpose |
|---|---|---|
| train | MOT20-01, MOT20-02 | YOLO training |
| val | MOT20-03 | Monitored during training |
| labeled_test | MOT20-05 | Final evaluation only — do not tune on this |
| test (no GT) | MOT20-04, 06, 07, 08 | Inference / visual inspection |

In [ ]:
!pip install ultralytics fiftyone sahi boxmot motmetrics

## 0. Environment setup

Set `ENVIRONMENT` to `"colab"` or `"local"` and fill in your paths.

In [ ]:
import sys, os

sys.path.insert(0, os.path.join(REPO_ROOT, "tracking", "src"))

from tracking_51 import (
    configure, Dataset,
    get_sequence_view,
    tracking_yolo, calc_metrics,
    save_results, save_sample_frames, save_annotated_video,
    train_detector,
)
import fiftyone as fo

# Apply path configuration — must be called before any Dataset/tracking function
configure(data_root=DATA_ROOT)

In [ ]:
import sys, os

sys.path.insert(0, os.path.join(REPO_ROOT, "tracking", "src"))

from tracking_51 import (
    configure, Dataset,
    get_sequence_view,
    tracking_yolo, calc_metrics, save_results,
    train_detector,
)
import fiftyone as fo

# Apply path configuration — must be called before any Dataset/tracking function
configure(data_root=DATA_ROOT)

---
## 1. Data preparation

Run once. Creates mp4 videos, builds the FiftyOne dataset, adds GT annotations,
and exports YOLO-format labels + images to `data/train_mot20/`.

In [ ]:
ds = Dataset()
fo_dataset = ds.prepare_all()

---
## 2. Train YOLO detector

Trains on MOT20-01 + MOT20-02, validates on MOT20-03.
Best weights are saved to `RUNS_DIR/train/weights/best.pt`.

In [ ]:
dataset_yaml = os.path.join(DATA_ROOT, "train_mot20", "dataset.yaml")

best_weights = train_detector(
    data_path=dataset_yaml,
    epochs=50,
    device=DEVICE,
    project=RUNS_DIR,
)
print(f"Best weights: {best_weights}")

---
## 3. Evaluate on validation set (MOT20-03)

Use this to tune hyperparameters (conf, iou thresholds).

In [ ]:
fo_dataset = fo.load_dataset("mot20")

# Use weights from training, or find the latest saved weights
if "best_weights" not in dir() or not os.path.exists(best_weights):
    import glob
    candidates = sorted(glob.glob(os.path.join(RUNS_DIR, "**/best.pt"), recursive=True))
    assert candidates, f"No best.pt found in {RUNS_DIR} — train first"
    best_weights = candidates[-1]
    print(f"Loaded saved weights: {best_weights}")

model = __import__("ultralytics").YOLO(best_weights)

val_view = get_sequence_view(fo_dataset, "MOT20-03")
tracking_yolo(val_view, model, tag="val_pred", device=DEVICE)

print("\n── Validation metrics (MOT20-03) ──")
val_summary = calc_metrics(val_view, tag="val_pred")

---
## 4. Final evaluation on labeled test set (MOT20-05)

Run this **only once** after all tuning is complete.
This is the held-out sequence — do not use its results to change any model decisions.

In [ ]:
test_view = get_sequence_view(fo_dataset, "MOT20-05")
tracking_yolo(test_view, model, tag="test_pred", device=DEVICE)

print("\n── Test metrics (MOT20-05) ──")
test_summary = calc_metrics(test_view, tag="test_pred")

if ENVIRONMENT == "local":
    session = fo.launch_app(fo_dataset)
    # session.wait()  # uncomment to block until the app is closed
else:
    # Save sample frames for val (for report)
    save_sample_frames(val_view, "val_pred")
    # Save full annotated video for test
    save_annotated_video(test_view, "test_pred")


In [ ]:
if ENVIRONMENT == "local":
    session = fo.launch_app(fo_dataset)
    # session.wait()  # uncomment to block until the app is closed
else:
    # Export annotated videos to Drive instead
    save_results(val_view,  "val_pred")
    save_results(test_view, "test_pred")